Run this notebook on [Serverless GPUs](https://docs.databricks.com/aws/en/compute/serverless/gpu). Serverless GPUs are in Beta as of the time of publishing, so please be aware that the APIs may change. Beta features are not production-ready.

In [0]:
%pip install ray folium
%pip install --extra-index-url=https://pypi.nvidia.com cuopt-server-cu12 cuopt-sh-client
%restart_python

In [0]:
%sh nvidia-smi

In [0]:
# import cudf
# from cuopt import routing

# # 4 nodes (0=depot, 1-3=tasks). Symmetric costs = travel times here.
# cost = cudf.DataFrame([[0,2,2,2],[2,0,2,2],[2,2,0,2],[2,2,2,0]], dtype='float32')
# n_locations = cost.shape[0]
# n_vehicles = 2
# n_orders = 3  # one order per task node

# dm = routing.DataModel(n_locations, n_vehicles, n_orders)
# dm.add_cost_matrix(cost)
# dm.add_transit_time_matrix(cost.copy(deep=True))  # separate if times differ

# ss = routing.SolverSettings()
# sol = routing.Solve(dm, ss)

# print(sol.get_route())      # pandas-like table
# sol.display_routes()        # pretty print

In [0]:
catalog = "default"
schema = "routing"
routing_table = "routing_unified_by_cluster"
routing_df = spark.read.table(f"{catalog}.{schema}.{routing_table}").where('cluster_id is not null')
display(routing_df)

In [0]:
# from serverless_gpu import distributed, runtime as rt
# from pyspark.sql.functions import col

# cluster_pdf = (
#     spark.read.table(f"{catalog}.{schema}.{routing_table}")
#     .withColumn("cluster_id", col("cluster_id").cast("string"))
#     .where("cluster_id = '1'")
#     .toPandas()
# )

# @distributed(gpus=1, gpu_type='a10', remote=True)
# def test(cluster_pdf):
#     return len(cluster_pdf)

# test.distributed(cluster_pdf)

In [0]:
# from serverless_gpu import distributed, runtime as rt

# @distributed(gpus=2, gpu_type='a10', remote=True)
# def test():
#     return rt.get_global_rank(), rt.get_local_rank(), rt.get_world_size()

# test.distributed()

In [0]:
import pandas as pd 


def add_missing_depot_arcs(pdf: pd.DataFrame) -> pd.DataFrame:
    # 1) find the depot's index from the data (it only appears as destination in your case)
    depot_idx = int(pdf.loc[pdf["destination_id"]=="DEPOT", "destination_index"].unique()[0])

    # 2) create reversed rows for arcs that go TO depot (i -> depot) so we also have (depot -> i)
    rev = (pdf.loc[pdf["destination_index"] == depot_idx,
                   ["origin_index","destination_index","duration_seconds"]]
             .rename(columns={
                 "origin_index": "destination_index",
                 "destination_index": "origin_index"
             }))

    # 3) drop any reversed rows that already exist
    rev = (rev.merge(pdf[["origin_index","destination_index"]],
                     on=["origin_index","destination_index"],
                     how="left", indicator=True)
              .loc[lambda d: d["_merge"]=="left_only",
                   ["origin_index","destination_index","duration_seconds"]])

    # 4) append and return
    return pd.concat([pdf, rev], ignore_index=True)

In [0]:
import math
import pandas as pd
import cudf
from cuopt import routing

SOLVER_THINKING_TIME = 0.1

def solve_single_route_from_matrix(pdf: pd.DataFrame) -> pd.DataFrame:
    # 1) Build N×N
    N = int(max(pdf["origin_index"].max(), pdf["destination_index"].max())) + 1
    BIG = 10**5
    pdf_aug = add_missing_depot_arcs(pdf)  # must include depot↔stops arcs

    M = pd.pivot_table(
        pdf_aug, index="origin_index", columns="destination_index",
        values="duration_seconds", aggfunc="min"
    ).reindex(index=range(N), columns=range(N))

    M = M.fillna(BIG).astype("float32")
    cost_mat = cudf.DataFrame(M.values)

    # 2) Solve in cuOpt (single vehicle; depot assumed 0)
    dm = routing.DataModel(N, 1, N - 1)
    dm.add_cost_matrix(cost_mat)

    ss = routing.SolverSettings()
    ss.set_time_limit(float(min(60, round(math.sqrt(N) * SOLVER_THINKING_TIME))))

    sol = routing.Solve(dm, ss)
    route_pd = sol.get_route().to_pandas()

    # 3) Robustly pick the node-id column
    loc_col = "location" if "location" in route_pd.columns else "route"

    # 4) Build (origin_index -> destination_index) edges in visit order
    route_pd = route_pd.sort_values(["truck_id", "arrival_stamp"]).reset_index(drop=True)
    route_pd["next_loc"] = route_pd.groupby("truck_id")[loc_col].shift(-1)

    edges = route_pd[route_pd["next_loc"].notna()].copy()
    edges["origin_index"] = edges[loc_col].astype(int)
    edges["destination_index"] = edges["next_loc"].astype(int)
    edges["route_index"] = edges.groupby("truck_id").cumcount()

    # 5) Deduplicate pdf_aug on arcs (match what went into the matrix)
    base = (
        pdf_aug
        .sort_values("duration_seconds")
        .drop_duplicates(["origin_index", "destination_index"], keep="first")
    )

    # 6) Join edges back to the original arcs to recover your columns
    merged = edges.merge(base, how="left", on=["origin_index", "destination_index"], suffixes=("", "_orig"))

    columns_to_select = ["route_index", "origin_id", "destination_id", "duration_seconds", "arrival_stamp", "latitude", 
                         "longitude", "origin_index", "destination_index"]
    out = (
        merged[columns_to_select]
        .sort_values(["route_index"])
        .reset_index(drop=True)
    )

    return out

In [0]:
pdf = routing_df.where("cluster_id = '1'").toPandas() # TODO not hardcoded just get the first one
solved_sample_cluster = solve_single_route_from_matrix(pdf)
display(solved_sample_cluster)

In [0]:
from utils.plotter import plot_route_folium
plot_route_folium(solved_sample_cluster[solved_sample_cluster['latitude'].notnull()])

In [0]:
import ray
from mlflow.utils.databricks_utils import get_databricks_env_vars
import os


def read_ray_dataset(catalog, schema, table):
    mlflow_dbrx_creds = get_databricks_env_vars("databricks")
    os.environ["DATABRICKS_HOST"] = mlflow_dbrx_creds['DATABRICKS_HOST']
    os.environ["DATABRICKS_TOKEN"] = mlflow_dbrx_creds['DATABRICKS_TOKEN']
    ds = ray.data.read_databricks_tables(
      warehouse_id='148ccb90800933a1',
      catalog=catalog,
      schema=schema,
      query=f'''
          SELECT CAST(cluster_id AS STRING) AS cluster_id, 
                * EXCEPT (cluster_id) 
          FROM {table}
          WHERE cluster_id is not null and cluster_id not like '%-%'
          ''',
    )
    return ds

In [0]:
from serverless_gpu.ray import ray_launch
import os, sys, logging


@ray_launch(gpus=4, gpu_type="a10", remote=True)
def optimize(catalog: str, schema: str, routing_table: str,
             volume: str = "ray_temp", out_subdir: str = "tempGPU") -> str:
    """
    Reads the routing table as a Ray Dataset, solves each cluster on a GPU,
    and writes the per-cluster results to Parquet under a Unity Catalog Volume.
    Returns the DBFS URI where results were written.
    """

    # 1) Environment: prefer UTF-8
    os.environ.setdefault("PYTHONIOENCODING", "UTF-8")
    os.environ.setdefault("LANG", "C.UTF-8")
    os.environ.setdefault("LC_ALL", "C.UTF-8")

    # 2) Reconfigure stdio encodings (Python 3.7+)
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    if hasattr(sys.stderr, "reconfigure"):
        sys.stderr.reconfigure(encoding="utf-8")

    # 3) Make every attached handler UTF-8-safe
    for name in ["", "ray", "ray.data", "ray.data._internal", "py4j", "databricks"]:
        logger = logging.getLogger(name)
        for h in list(logger.handlers):
            try:
                if hasattr(h, "stream") and hasattr(h.stream, "reconfigure"):
                    h.stream.reconfigure(encoding="utf-8")
            except Exception:
                pass
    ray_ds = read_ray_dataset(catalog, schema, routing_table)

    solved = (
        ray_ds
        .groupby("cluster_id")
        .map_groups(
            solve_single_route_from_matrix,
            batch_format="pandas",
            num_gpus=1
        )
    )

    tmp_dir_uri = f"/Volumes/{catalog}/{schema}/{volume}/{out_subdir}"
    solved.write_parquet(tmp_dir_uri)

    return tmp_dir_uri

In [0]:
optimize.distributed(catalog, schema, routing_table)